# Loan Approval Classification using Apache Spark

**Project:** Loan Approval Classification  
**Framework:** Apache Spark / PySpark  
**Dataset:** Kaggle Loan Approval Classification Data

This notebook implements the complete workflow: data ingestion, exploratory data analysis, preprocessing, feature engineering, model training, evaluation, and feature-importance analysis.


## 1. Setup and Data Loading

The original project was designed for Databricks/Spark. For portability, `DATA_PATH` can be changed to the local location of `loan_data.csv`.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, mean, stddev, min, max
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

spark = (
    SparkSession.builder
    .appName("LoanApprovalClassification")
    .getOrCreate()
)

# Change this path if running outside Databricks.
DATA_PATH = "/FileStore/tables/loan_data.csv"

loan_data = spark.read.csv(
    DATA_PATH,
    header=True,
    inferSchema=True
)

print(f"Rows: {loan_data.count():,}")
print(f"Columns: {len(loan_data.columns)}")
loan_data.printSchema()
loan_data.show(5, truncate=False)


## 2. Dataset Overview

The project uses approximately **45,000 loan applications** and **14 variables** covering demographic, financial and loan-related information. The target is `loan_status`.


In [ ]:
# Basic descriptive statistics
loan_data.describe().show()

# Column names
print("Columns:")
for c in loan_data.columns:
    print("-", c)


## 3. Exploratory Data Analysis (EDA)

The analysis covers:
- Numerical distributions
- Categorical distributions
- Outlier inspection
- Missing-value analysis
- Duplicate detection
- Loan-status distribution
- Relationships between categorical variables and loan status


In [ ]:
numerical_columns = [
    "person_age", "person_income", "person_emp_exp", "loan_amnt",
    "loan_int_rate", "loan_percent_income",
    "cb_person_cred_hist_length", "credit_score"
]

for column in numerical_columns:
    pdf = loan_data.select(column).toPandas()
    plt.figure(figsize=(8, 5))
    plt.hist(pdf[column].dropna(), bins=20, edgecolor="black")
    plt.title(f"Histogram for {column}")
    plt.xlabel(column)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
# Missing-value analysis
missing_values = loan_data.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in loan_data.columns
])
missing_values.show()

# Duplicate rows
duplicate_count = loan_data.count() - loan_data.dropDuplicates().count()
print(f"Duplicate rows: {duplicate_count:,}")

# Remove duplicates if present
loan_data = loan_data.dropDuplicates()
print(f"Rows after duplicate removal: {loan_data.count():,}")


In [ ]:
# Inspect unusually high income values.
# The original notebook specifically investigated income above 10 lakh.
high_income = loan_data.filter(col("person_income") > 1_000_000)
print(f"Rows with income > 1,000,000: {high_income.count():,}")
high_income.show(10, truncate=False)


In [ ]:
# Outlier inspection for age and income using a sample
sample_pdf = (
    loan_data
    .select("person_age", "person_income")
    .sample(fraction=0.10, seed=42)
    .limit(5000)
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(x=sample_pdf["person_age"], ax=axes[0])
axes[0].set_title("Outlier Detection: Person Age")
axes[0].set_xlabel("Age")

sns.boxplot(x=sample_pdf["person_income"], ax=axes[1])
axes[1].set_title("Outlier Detection: Person Income")
axes[1].set_xlabel("Income")

plt.tight_layout()
plt.show()


In [ ]:
# Loan status distribution
loan_status_distribution = (
    loan_data.groupBy("loan_status")
    .count()
    .orderBy("loan_status")
    .toPandas()
)

loan_status_distribution["percentage"] = (
    loan_status_distribution["count"]
    / loan_status_distribution["count"].sum()
    * 100
)

display(loan_status_distribution)

plt.figure(figsize=(8, 5))
plt.bar(
    loan_status_distribution["loan_status"].astype(str),
    loan_status_distribution["count"],
    edgecolor="black"
)
plt.title("Loan Status Distribution")
plt.xlabel("Loan Status")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


In [ ]:
# Categorical distributions by loan status
categorical_columns = [
    "person_gender",
    "person_education",
    "person_home_ownership",
    "loan_intent"
]

loan_data_pd = loan_data.toPandas()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, column in enumerate(categorical_columns):
    data = loan_data_pd.groupby([column, "loan_status"]).size().unstack(fill_value=0)
    categories = data.index.astype(str)
    x = np.arange(len(categories))
    width = 0.4

    if 0 in data.columns:
        axes[i].bar(x - width/2, data[0], width, label="Loan Status 0")
    if 1 in data.columns:
        axes[i].bar(x + width/2, data[1], width, label="Loan Status 1")

    axes[i].set_title(f"Loan Status by {column}")
    axes[i].set_xlabel(column)
    axes[i].set_ylabel("Count")
    axes[i].set_xticks(x)
    axes[i].set_xticklabels(categories, rotation=45, ha="right")
    axes[i].legend()

plt.tight_layout()
plt.show()


## 4. Feature Engineering and Preprocessing

The original workflow uses:
- Binary encoding for gender and previous loan defaults
- String indexing for education, home ownership and loan intent
- One-hot encoding objects for selected categorical variables
- Median replacement for implausible ages above 100
- Vector assembly
- Standard scaling

The model feature vector follows the feature list used in the original notebook.


In [ ]:
# Binary encodings
loan_data = loan_data.withColumn(
    "person_gender_binary",
    F.when(F.col("person_gender") == "Male", 1).otherwise(0)
)

loan_data = loan_data.withColumn(
    "previous_loan_defaults_binary",
    F.when(F.col("previous_loan_defaults_on_file") == "Yes", 1).otherwise(0)
)

# Education indexing
education_indexer = StringIndexer(
    inputCol="person_education",
    outputCol="person_education_ordinal",
    handleInvalid="skip"
)
loan_data = education_indexer.fit(loan_data).transform(loan_data)

# Index categorical variables used in the model
indexer_home = StringIndexer(
    inputCol="person_home_ownership",
    outputCol="person_home_ownership_index",
    handleInvalid="keep"
)
indexer_intent = StringIndexer(
    inputCol="loan_intent",
    outputCol="loan_intent_index",
    handleInvalid="keep"
)

loan_data = indexer_home.fit(loan_data).transform(loan_data)
loan_data = indexer_intent.fit(loan_data).transform(loan_data)

# Create one-hot encoded versions for exploratory/feature-engineering completeness.
encoder = OneHotEncoder(
    inputCols=["person_home_ownership_index", "loan_intent_index"],
    outputCols=["person_home_ownership_encoded", "loan_intent_encoded"]
)
loan_data = encoder.fit(loan_data).transform(loan_data)

loan_data.select(
    "person_gender", "person_gender_binary",
    "previous_loan_defaults_on_file", "previous_loan_defaults_binary",
    "person_education", "person_education_ordinal",
    "person_home_ownership", "person_home_ownership_encoded",
    "loan_intent", "loan_intent_encoded"
).show(truncate=False)


In [ ]:
# Replace implausible age values (>100) with the dataset median
median_person_age = loan_data.approxQuantile(
    "person_age", [0.5], 0.01
)[0]

loan_data = loan_data.withColumn(
    "person_age",
    F.when(F.col("person_age") > 100, median_person_age)
     .otherwise(F.col("person_age"))
)

print("Median age used for replacement:", median_person_age)


In [ ]:
# Correlation analysis
correlation_columns = [
    "person_age", "person_income", "person_emp_exp", "loan_amnt",
    "loan_int_rate", "loan_percent_income",
    "cb_person_cred_hist_length", "credit_score",
    "loan_status", "person_gender_binary",
    "previous_loan_defaults_binary", "person_education_ordinal",
    "person_home_ownership_index", "loan_intent_index"
]

correlation_matrix = loan_data.select(correlation_columns).toPandas().corr()

plt.figure(figsize=(12, 9))
sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    cbar=True
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

loan_status_correlation = (
    correlation_matrix["loan_status"]
    .sort_values(ascending=False)
    .to_frame("Correlation")
)

display(loan_status_correlation)


### Correlation observations from the original analysis

- `person_income` and `loan_amnt` show a positive correlation of about **0.59**.
- `previous_loan_defaults_binary` and `loan_status` show a negative correlation of about **-0.54**.
- `loan_percent_income` and `loan_status` show a positive correlation of about **0.38**.
- `loan_int_rate` and `loan_status` show a positive correlation of about **0.33**.

Correlation indicates association, not causation.


## 5. Train–Validation Split and Feature Vector

The original project uses an **80:20 split** with a fixed random seed of **42**.


In [ ]:
target_column = "loan_status"

feature_columns = [
    "person_age", "person_income", "person_emp_exp", "loan_amnt",
    "loan_int_rate", "loan_percent_income",
    "cb_person_cred_hist_length", "credit_score",
    "person_gender_binary", "previous_loan_defaults_binary",
    "person_education_ordinal", "person_home_ownership_index",
    "loan_intent_index"
]

selected_columns = feature_columns + [target_column]
loan_data_selected = loan_data.select(*selected_columns)

train_data, validation_data = loan_data_selected.randomSplit(
    [0.8, 0.2],
    seed=42
)

print(f"Training Data Count: {train_data.count():,}")
print(f"Validation Data Count: {validation_data.count():,}")

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

train_data = assembler.transform(train_data)
validation_data = assembler.transform(validation_data)

scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(train_data)

train_data_scaled = scaler_model.transform(train_data)
validation_data_scaled = scaler_model.transform(validation_data)

train_data_final = train_data_scaled.select(
    col("scaled_features"),
    col(target_column).alias("label")
)

validation_data_final = validation_data_scaled.select(
    col("scaled_features"),
    col(target_column).alias("label")
)

train_data_final.show(5, truncate=False)


## 6. Logistic Regression

Logistic Regression is used as an interpretable baseline classification model.


In [ ]:
logistic_regression = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="label"
)

lr_model = logistic_regression.fit(train_data_final)
lr_predictions = lr_model.transform(validation_data_final)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

lr_accuracy = accuracy_evaluator.evaluate(lr_predictions)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)
lr_f1 = f1_evaluator.evaluate(lr_predictions)

print(f"Logistic Regression Accuracy: {lr_accuracy:.2f}")
print(f"Logistic Regression F1 Score: {lr_f1:.2f}")

lr_predictions.select(
    "label", "prediction", "probability"
).show(5, truncate=False)


In [ ]:
# Logistic Regression confusion matrix
lr_confusion = (
    lr_predictions.groupBy("label")
    .pivot("prediction")
    .count()
    .fillna(0)
)

lr_confusion.show()

lr_confusion_pd = lr_confusion.toPandas().set_index("label")

plt.figure(figsize=(8, 6))
sns.heatmap(lr_confusion_pd, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


## 7. Random Forest Classifier

Random Forest is used to capture non-linear relationships among borrower, credit-history and loan characteristics.


In [ ]:
rf_classifier = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="label",
    numTrees=100,
    maxDepth=10,
    seed=42
)

rf_model = rf_classifier.fit(train_data_final)
rf_predictions = rf_model.transform(validation_data_final)

rf_accuracy = accuracy_evaluator.evaluate(rf_predictions)
rf_f1 = f1_evaluator.evaluate(rf_predictions)

print(f"Random Forest Accuracy: {rf_accuracy:.2f}")
print(f"Random Forest F1 Score: {rf_f1:.2f}")

rf_predictions.select(
    "label", "prediction", "probability"
).show(5, truncate=False)


In [ ]:
# Random Forest confusion matrix
rf_confusion = (
    rf_predictions.groupBy("label")
    .pivot("prediction")
    .count()
    .fillna(0)
)

rf_confusion.show()

rf_confusion_pd = rf_confusion.toPandas().set_index("label")

plt.figure(figsize=(8, 6))
sns.heatmap(rf_confusion_pd, annot=True, fmt="d", cmap="Blues", cbar=True)
plt.title("Confusion Matrix - Random Forest")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


In [ ]:
# Random Forest feature importance
feature_importances = rf_model.featureImportances.toArray()

feature_importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": feature_importances
}).sort_values("Importance", ascending=False)

display(feature_importance_df)

plt.figure(figsize=(10, 6))
plt.barh(
    feature_importance_df["Feature"],
    feature_importance_df["Importance"]
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance - Random Forest")
plt.tight_layout()
plt.show()


## 8. Model Comparison and Final Result

The project documentation reports the following validation accuracies:

| Model | Accuracy |
|---|---:|
| Logistic Regression | 0.89 |
| Random Forest | 0.92 |

The reported Random Forest result is approximately **3 percentage points higher** than the Logistic Regression baseline.


In [ ]:
# Recalculate model comparison from the current run
model_comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "Accuracy": [lr_accuracy, rf_accuracy],
    "F1 Score": [lr_f1, rf_f1]
})

display(model_comparison)

print("Model comparison completed.")


## 9. Conclusion

This project demonstrates an end-to-end scalable machine-learning workflow with PySpark, from data ingestion and EDA through feature engineering, classification and evaluation. The project documentation reports Random Forest as the higher-accuracy model on the validation data.

**Important:** The reported accuracy values are validation results from the project; they should not be interpreted as a guarantee of real-world lending performance.


In [ ]:
# Optional: stop Spark when the notebook is finished
# spark.stop()
